# 05 - Análisis de la variable alcohol

**Análisis posterior a la evaluación final** (`docs/protocolo_analisis.md`, sección 8.1). Usa **solo datos 2021–2024** y no modifica el modelo congelado.

**Motivo.** En la API, para una persona que consumió alcohol en los últimos 12 meses, esta variable contribuye a *disminuir* la probabilidad estimada. Antes de interpretarlo se verifica:

1. La pregunta `QS208` y su codificación en los cuestionarios y diccionarios 2021–2025.
2. Que la API codifique la respuesta igual que el entrenamiento.
3. Qué explica la contribución: clase positiva, signo y valor de referencia.
4. La asociación observada, en conjunto, por edad, sexo y año, y ajustada por otros predictores.
5. Con validación cruzada (mismas particiones), si el alcohol aporta al modelo web.

No se asume de antemano que el resultado se deba a cambios de hábitos después del diagnóstico.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from preparacion import DESENLACE, PREDICTORES_COMPLETO, PREDICTORES_WEB, RUTA_DESARROLLO, codificar

sys.path.insert(0, str(Path('../backend').resolve()))
from app.schemas.diabetes import EvaluacionEntrada  # noqa: E402
from app.services.predictor import ModeloRiesgo  # noqa: E402

pd.set_option('display.float_format', '{:.4f}'.format)
SEMILLA = 42
SALIDA = Path('../resultados/05_alcohol')
SALIDA.mkdir(parents=True, exist_ok=True)

crudo = pd.read_csv(RUTA_DESARROLLO)
d = codificar(crudo)
assert d['anio'].max() <= 2024
print(f'Desarrollo 2021-2024: {len(d)} registros, {int(d[DESENLACE].sum())} con diagnóstico autorreportado')

Desarrollo 2021-2024: 8909 registros, 562 con diagnóstico autorreportado


## 1. Pregunta y codificación de `QS208`

Verificación en los cuestionarios de salud y diccionarios de variables 2021–2025 (archivos del INEI en `ENDES/<año>/1640/`):

| Pregunta | Texto (idéntico en los cinco años) | Códigos | Salto |
|---|---|---|---|
| `QS206` | ¿Alguna vez en su vida usted ha consumido alguna bebida alcohólica o licor? | 1 Sí, 2 No, 8 No sabe/no recuerda | Si «No», pasa a la pregunta 213 (no se pregunta `QS208`) |
| `QS208` | ¿En los últimos 12 meses, es decir, desde ___ del año pasado hasta ___ de este año, usted ha consumido alguna bebida alcohólica o licor? | 1 Sí, 2 No, 8 No sabe/no recuerda | Si «No», pasa a la pregunta 213 |

- En 2025 se añadió una instrucción al encuestador en `QS206` («indague si consumió al menos un vaso / copa o unidad similar completa»). 2025 no interviene en este análisis.
- En los diccionarios PDF de 2021 y 2022, la descripción de `QS208` aparece como «Tomó cerveza». Es un **desalineamiento de la maquetación del PDF**: los textos aparecen desplazados respecto de las filas de códigos, y la descripción «En los últimos 12 meses ha consumido alguna bebida alcohólica» figura en la fila contigua. Los datos lo confirman: `QS208` está vacía exactamente cuando `QS206` ≠ 1, que es el salto del cuestionario.

**Codificación usada en el entrenamiento** (`notebooks/preparacion.py`): `alcohol` = 1 si `QS208` = 1; 0 si `QS208` = 2 o si `QS206` = 2 (nunca bebió, por el salto); faltante si «no sabe» (`QS206` = 8 o `QS208` = 8), que luego se imputa con la moda dentro del pipeline.

In [2]:
verificacion = pd.crosstab(
    [crudo['ANIO'], crudo['QS206'].fillna(-1).astype(int).rename('QS206')],
    crudo['QS208'].fillna(-1).astype(int).rename('QS208 (-1 = vacía)'),
)
verificacion.to_csv(SALIDA / 'qs206_qs208_por_anio.csv')
verificacion

QS208 (-1 = vacía)  -1     1    2
ANIO QS206                       
2021 1               0  1700  602
     2              55     0    0
2022 1               0  1767  435
     2              53     0    0
     8               1     0    0
2023 1               0  1718  381
     2              36     0    0
2024 1               0  1682  425
     2              54     0    0

In [3]:
control = pd.crosstab(
    [crudo['QS206'].fillna(-1).astype(int).rename('QS206'), crudo['QS208'].fillna(-1).astype(int).rename('QS208')],
    d['alcohol'].fillna(-1).astype(int).rename('alcohol codificado (-1 = faltante)'),
)
control

alcohol codificado (-1 = faltante)  -1     0     1
QS206 QS208                                       
1      1                             0     0  6867
       2                             0  1843     0
2     -1                             0   198     0
8     -1                             1     0     0

**Resultado.** En los cuatro años, `QS208` solo está vacía cuando `QS206` = 2 (nunca bebió) o `QS206` = 8; la proporción de «sí» es similar entre años (77%–82% de quienes bebieron alguna vez). La variable codificada reproduce las reglas: los 198 que nunca bebieron quedan como 0 y el único «no sabe» queda como faltante. **No hay error de codificación en el entrenamiento.**

## 2. Integración con la API

Se pasan por la API (esquema `EvaluacionEntrada` y clase `ModeloRiesgo` del backend) todos los registros de 2021–2024 que tienen los nueve predictores completos y valores dentro de los rangos del formulario, y se compara la fila construida por la API y su probabilidad con las del entrenamiento.

In [4]:
artefacto = joblib.load('../models/v2/web_regresion_logistica.joblib')
pipeline = artefacto['modelo']
modelo_api = ModeloRiesgo('../models/v2')

filas_api, prob_api, idx_ok, rechazados = [], [], [], 0
for i, r in d.dropna(subset=PREDICTORES_WEB).iterrows():
    try:
        entrada = EvaluacionEntrada(
            edad=int(r['edad']), sexo='masculino' if r['hombre'] == 1 else 'femenino',
            peso_kg=crudo.at[i, 'QS900'], talla_cm=crudo.at[i, 'QS901'], cintura_cm=r['cintura'],
            hipertension=bool(r['hta']), fuma=bool(r['tabaco']), alcohol=bool(r['alcohol']),
            dias_frutas=int(r['frutas_dias']), dias_verduras=int(r['verduras_dias']),
        )
    except ValueError:
        rechazados += 1
        continue
    filas_api.append(modelo_api._fila(entrada).iloc[0])
    prob_api.append(modelo_api.predecir(entrada).probabilidad)
    idx_ok.append(i)

filas_api = pd.DataFrame(filas_api, index=idx_ok)
prob_entrenamiento = pipeline.predict_proba(d.loc[idx_ok, PREDICTORES_WEB])[:, 1]
print(f'Registros comparados: {len(idx_ok)} (fuera de los rangos del formulario: {rechazados})')
print('Coincidencia exacta de la variable alcohol (API vs. entrenamiento):',
      bool((filas_api['alcohol'] == d.loc[idx_ok, 'alcohol']).all()))
print('Diferencias de codificación por variable (excepto IMC):',
      {v: int((filas_api[v] != d.loc[idx_ok, v]).sum()) for v in PREDICTORES_WEB if v != 'imc'})
print(f'Diferencia máxima en el IMC (la API lo redondea a 2 decimales): {np.abs(filas_api["imc"] - d.loc[idx_ok, "imc"]).max():.4f}')
print(f'Diferencia máxima de probabilidad API vs. entrenamiento: {np.abs(np.array(prob_api) - prob_entrenamiento).max():.6f}')

Registros comparados: 8897 (fuera de los rangos del formulario: 0)
Coincidencia exacta de la variable alcohol (API vs. entrenamiento): True
Diferencias de codificación por variable (excepto IMC): {'edad': 0, 'hombre': 0, 'cintura': 0, 'hta': 0, 'tabaco': 0, 'alcohol': 0, 'frutas_dias': 0, 'verduras_dias': 0}
Diferencia máxima en el IMC (la API lo redondea a 2 decimales): 0.0050
Diferencia máxima de probabilidad API vs. entrenamiento: 0.000085


## 3. Qué explica la contribución del alcohol

La contribución que muestra la API es el coeficiente de la regresión logística por el valor estandarizado de la variable. Se verifica a qué clase corresponde, su signo y respecto de qué valor de referencia se mide.

In [5]:
lr = pipeline.named_steps['modelo']
prep = pipeline.named_steps['prep']
columnas = [n.split('__', 1)[1] for n in prep.get_feature_names_out()]
escalador = prep.named_transformers_['categoricas'].named_steps['escalar']
cats = prep.transformers_[1][2]
j = cats.index('alcohol')
media, desv = escalador.mean_[j], escalador.scale_[j]
coef_std = lr.coef_[0][columnas.index('alcohol')]
coef = coef_std / desv

print('Clases del modelo:', lr.classes_, '-> predict_proba[:, 1] = P(QS109 = diagnóstico reportado)')
print(f'Coeficiente del alcohol: {coef:.4f} por (sí vs. no) -> OR = {np.exp(coef):.3f}')
print(f'Valor de referencia (media de alcohol en 2021-2024): {media:.3f} (proporción que bebió en los últimos 12 meses)')
print(f'Contribución si bebió (alcohol = 1): {coef_std * (1 - media) / desv:+.4f} log-odds')
print(f'Contribución si no bebió (alcohol = 0): {coef_std * (0 - media) / desv:+.4f} log-odds')

Clases del modelo: [0 1] -> predict_proba[:, 1] = P(QS109 = diagnóstico reportado)
Coeficiente del alcohol: -0.1990 por (sí vs. no) -> OR = 0.820
Valor de referencia (media de alcohol en 2021-2024): 0.771 (proporción que bebió en los últimos 12 meses)
Contribución si bebió (alcohol = 1): -0.0456 log-odds
Contribución si no bebió (alcohol = 0): +0.1534 log-odds


**Interpretación.** El modelo explica la clase positiva (diagnóstico reportado). La contribución se mide **respecto de la persona promedio de 2021–2024**, en la que la variable alcohol vale ≈ 0.77. Por eso, para quien bebió, la contribución es negativa (está por encima del promedio y el coeficiente es negativo), y para quien no bebió es positiva. Esta contribución describe cómo se mueve la probabilidad estimada de *esa* persona respecto del promedio dentro del modelo; **no demuestra un efecto protector del alcohol**. El signo corresponde a la asociación condicional que el modelo aprendió de los datos, que se examina a continuación.

## 4. Asociación observada en 2021–2024

### 4.1 En conjunto y según consumo de alcohol a lo largo de la vida

Se distinguen tres grupos con `QS206` y `QS208`: nunca bebió, bebió alguna vez pero no en los últimos 12 meses, y bebió en los últimos 12 meses. Proporciones sin ponderar y ponderadas.

In [6]:
def media_ponderada(x, w):
    ok = x.notna() & w.notna()  # algunos registros no tienen factor de ponderación
    return float(np.average(x[ok], weights=w[ok]))


def tabla_proporcion(datos, grupos):
    filas = []
    for clave, g in datos.groupby(grupos, observed=True):
        clave = clave if isinstance(clave, tuple) else (clave,)
        filas.append({**dict(zip(grupos, clave)), 'n': len(g), 'casos': int(g[DESENLACE].sum()),
                      '%_diagnostico': 100 * g[DESENLACE].mean(),
                      '%_ponderado': 100 * media_ponderada(g[DESENLACE], g['peso_muestral'])})
    return pd.DataFrame(filas)


d['alcohol_12m'] = d['alcohol'].map({1.0: 'bebió en 12 meses', 0.0: 'no bebió en 12 meses'})
d['consumo_vida'] = np.select(
    [crudo['QS206'] == 2, (crudo['QS206'] == 1) & (crudo['QS208'] == 2), crudo['QS208'] == 1],
    ['nunca bebió', 'bebió alguna vez, no en 12 meses', 'bebió en 12 meses'], default=None)
d['grupo_edad'] = pd.cut(d['edad'], [29, 39, 49, 59, 69], labels=['30-39', '40-49', '50-59', '60-69'])
d['sexo'] = np.where(d['hombre'] == 1, 'hombre', 'mujer')

general = tabla_proporcion(d.dropna(subset=['alcohol_12m']), ['alcohol_12m'])
vida = tabla_proporcion(d.dropna(subset=['consumo_vida']), ['consumo_vida'])
general.to_csv(SALIDA / 'proporcion_general.csv', index=False)
vida.to_csv(SALIDA / 'proporcion_consumo_vida.csv', index=False)
display(general)
vida

,alcohol_12m,n,casos,%_diagnostico,%_ponderado
0,bebió en 12 meses,6867,379,5.5191,7.0819
1,no bebió en 12 meses,2041,183,8.9662,11.2545


,consumo_vida,n,casos,%_diagnostico,%_ponderado
0,"bebió alguna vez, no en 12 meses",1843,168,9.1156,11.6997
1,bebió en 12 meses,6867,379,5.5191,7.0819
2,nunca bebió,198,15,7.5758,6.7474


### 4.2 Por grupo de edad, por sexo y por año

In [7]:
por_edad = tabla_proporcion(d.dropna(subset=['alcohol_12m']), ['grupo_edad', 'alcohol_12m'])
por_sexo = tabla_proporcion(d.dropna(subset=['alcohol_12m']), ['sexo', 'alcohol_12m'])
por_anio = tabla_proporcion(d.dropna(subset=['alcohol_12m']), ['anio', 'alcohol_12m'])
por_edad.to_csv(SALIDA / 'proporcion_por_edad.csv', index=False)
por_sexo.to_csv(SALIDA / 'proporcion_por_sexo.csv', index=False)
por_anio.to_csv(SALIDA / 'proporcion_por_anio.csv', index=False)
display(por_edad)
display(por_sexo)
por_anio

,grupo_edad,alcohol_12m,n,casos,%_diagnostico,%_ponderado
0,30-39,bebió en 12 meses,3378,66,1.9538,2.0329
1,30-39,no bebió en 12 meses,795,22,2.7673,2.5074
2,40-49,bebió en 12 meses,1822,82,4.5005,4.9020
3,40-49,no bebió en 12 meses,452,23,5.0885,5.8596
4,50-59,bebió en 12 meses,965,117,12.1244,12.2365
5,50-59,no bebió en 12 meses,373,51,13.6729,14.2688
6,60-69,bebió en 12 meses,702,114,16.2393,17.2501
7,60-69,no bebió en 12 meses,421,87,20.6651,22.0172


,sexo,alcohol_12m,n,casos,%_diagnostico,%_ponderado
0,hombre,bebió en 12 meses,3386,175,5.1683,6.4173
1,hombre,no bebió en 12 meses,648,54,8.3333,12.1257
2,mujer,bebió en 12 meses,3481,204,5.8604,7.7927
3,mujer,no bebió en 12 meses,1393,129,9.2606,10.7928


,anio,alcohol_12m,n,casos,%_diagnostico,%_ponderado
0,2021,bebió en 12 meses,1700,89,5.2353,7.0245
1,2021,no bebió en 12 meses,657,53,8.0670,9.8889
2,2022,bebió en 12 meses,1767,96,5.4329,6.7646
3,2022,no bebió en 12 meses,488,40,8.1967,9.7543
4,2023,bebió en 12 meses,1718,111,6.4610,8.4262
5,2023,no bebió en 12 meses,417,44,10.5516,12.4701
6,2024,bebió en 12 meses,1682,83,4.9346,6.0664
7,2024,no bebió en 12 meses,479,46,9.6033,13.1180


In [8]:
bebe_por_edad = d.groupby(['grupo_edad', 'sexo'], observed=True)['alcohol'].mean().mul(100).unstack().round(1)
print('Porcentaje que bebió en los últimos 12 meses, por edad y sexo:')
bebe_por_edad

Porcentaje que bebió en los últimos 12 meses, por edad y sexo:


sexo,hombre,mujer
grupo_edad,,
30-39,87.8000,75.4000
40-49,86.2000,74.3000
50-59,78.8000,66.8000
60-69,69.5000,57.5000


### 4.3 Odds ratios del alcohol ajustando progresivamente por otros predictores

Regresión logística en casos completos. Edad por cada 10 años, IMC por cada 5 kg/m² y perímetro abdominal por cada 10 cm, como en el notebook 02.

In [9]:
m = d.dropna(subset=PREDICTORES_WEB).copy()
m['edad10'], m['imc5'], m['cintura10'] = m['edad'] / 10, m['imc'] / 5, m['cintura'] / 10
y_ = DESENLACE

especificaciones = {
    'crudo': 'alcohol',
    '+ edad y sexo': 'alcohol + edad10 + hombre',
    '+ IMC, perímetro abdominal e hipertensión': 'alcohol + edad10 + hombre + imc5 + cintura10 + hta',
    'modelo web completo': 'alcohol + edad10 + hombre + imc5 + cintura10 + hta + tabaco + frutas_dias + verduras_dias',
}
filas = []
for nombre, rhs in especificaciones.items():
    ajuste = smf.logit(f'{y_} ~ {rhs}', data=m).fit(disp=False)
    ic = ajuste.conf_int().loc['alcohol']
    filas.append({'modelo': nombre, 'OR_alcohol': np.exp(ajuste.params['alcohol']),
                  'IC95_inf': np.exp(ic[0]), 'IC95_sup': np.exp(ic[1]), 'p': ajuste.pvalues['alcohol']})
or_progresivo = pd.DataFrame(filas)
or_progresivo.to_csv(SALIDA / 'or_alcohol_ajuste_progresivo.csv', index=False)
or_progresivo

,modelo,OR_alcohol,IC95_inf,IC95_sup,p
0,crudo,0.5960,0.4958,0.7165,0.0000
1,+ edad y sexo,0.8449,0.6951,1.0270,0.0906
2,"+ IMC, perímetro abdominal e hipertensión",0.8434,0.6909,1.0296,0.0943
3,modelo web completo,0.8263,0.6744,1.0124,0.0656


### 4.4 Consumo a lo largo de la vida, ajustado

Mismo ajuste que el modelo web, con la categoría «nunca bebió» como referencia.

In [10]:
m['consumo_vida'] = d.loc[m.index, 'consumo_vida']
ajuste_vida = smf.logit(
    f"{y_} ~ C(consumo_vida, Treatment('nunca bebió')) + edad10 + hombre + imc5 + cintura10 + hta + tabaco + frutas_dias + verduras_dias",
    data=m.dropna(subset=['consumo_vida'])).fit(disp=False)
terminos = [t for t in ajuste_vida.params.index if t.startswith('C(consumo_vida')]
or_vida = pd.DataFrame({
    'categoria (vs. nunca bebió)': [t.split('[T.')[1].rstrip(']') for t in terminos],
    'OR': np.exp(ajuste_vida.params[terminos]).values,
    'IC95_inf': np.exp(ajuste_vida.conf_int().loc[terminos, 0]).values,
    'IC95_sup': np.exp(ajuste_vida.conf_int().loc[terminos, 1]).values,
    'p': ajuste_vida.pvalues[terminos].values,
})
or_vida.to_csv(SALIDA / 'or_consumo_vida_ajustado.csv', index=False)
or_vida

,categoria (vs. nunca bebió),OR,IC95_inf,IC95_sup,p
0,"bebió alguna vez, no en 12 meses",0.9846,0.5462,1.7749,0.9588
1,bebió en 12 meses,0.8146,0.4574,1.4508,0.4862


### 4.5 ¿Cambia la asociación entre años, grupos de edad o sexo?

Prueba de razón de verosimilitud de la interacción del alcohol con el año, el grupo de edad y el sexo, sobre el modelo web ajustado.

In [11]:
m['anio_c'] = m['anio'].astype(str)
m['grupo_edad'] = d.loc[m.index, 'grupo_edad'].astype(str)
base = 'alcohol + edad10 + hombre + imc5 + cintura10 + hta + tabaco + frutas_dias + verduras_dias'
filas = []
for nombre, extra, interaccion in [('año', 'C(anio_c)', 'alcohol:C(anio_c)'),
                                    ('grupo de edad', 'C(grupo_edad)', 'alcohol:C(grupo_edad)'),
                                    ('sexo', '', 'alcohol:hombre')]:
    sin = smf.logit(f'{y_} ~ {base}' + (f' + {extra}' if extra else ''), data=m).fit(disp=False)
    con = smf.logit(f'{y_} ~ {base}' + (f' + {extra}' if extra else '') + f' + {interaccion}', data=m).fit(disp=False)
    lr_stat = 2 * (con.llf - sin.llf)
    gl = con.df_model - sin.df_model
    filas.append({'interacción del alcohol con': nombre, 'chi2': lr_stat, 'gl': int(gl), 'p': stats.chi2.sf(lr_stat, gl)})
interacciones = pd.DataFrame(filas)
interacciones.to_csv(SALIDA / 'interacciones.csv', index=False)
interacciones

,interacción del alcohol con,chi2,gl,p
0,año,2.4638,3,0.4819
1,grupo de edad,2.0157,3,0.5692
2,sexo,0.0119,1,0.9130


### 4.6 ¿Por qué el notebook 02 reporta p = 0.041 y este análisis p = 0.07?

El notebook 02 estima el OR del alcohol en el **modelo completo** (14 predictores: los 9 del modelo web más presión arterial sistólica y diastólica, nivel educativo, seguro de salud e índice de riqueza). Este notebook lo estima con los **9 predictores del modelo web**. Para identificar el origen de la diferencia, se reestima el modelo web sobre la misma muestra del notebook 02 y se agrega cada variable adicional por separado.

In [12]:
mc = d.dropna(subset=PREDICTORES_COMPLETO).copy()
mc['edad10'], mc['imc5'], mc['cintura10'] = mc['edad'] / 10, mc['imc'] / 5, mc['cintura'] / 10
mc['pas10'], mc['pad10'] = mc['pas'] / 10, mc['pad'] / 10
web = 'alcohol + edad10 + hombre + imc5 + cintura10 + hta + tabaco + frutas_dias + verduras_dias'

especificaciones = [
    ('Notebook 02: modelo completo (14 predictores)', web + ' + pas10 + pad10 + educacion + seguro + riqueza', mc),
    ('Este notebook: predictores del modelo web (9)', web, m),
    ('Predictores web, misma muestra del notebook 02', web, mc),
    ('Web + presión arterial', web + ' + pas10 + pad10', mc),
    ('Web + nivel educativo', web + ' + educacion', mc),
    ('Web + seguro de salud', web + ' + seguro', mc),
    ('Web + índice de riqueza', web + ' + riqueza', mc),
]
filas = []
for nombre, rhs, datos in especificaciones:
    ajuste = smf.logit(f'{y_} ~ {rhs}', data=datos).fit(disp=False)
    ic = ajuste.conf_int().loc['alcohol']
    filas.append({'modelo': nombre, 'n': len(datos), 'OR_alcohol': np.exp(ajuste.params['alcohol']),
                  'IC95_inf': np.exp(ic[0]), 'IC95_sup': np.exp(ic[1]), 'p': ajuste.pvalues['alcohol']})
completo_vs_web = pd.DataFrame(filas)
completo_vs_web.to_csv(SALIDA / 'or_alcohol_completo_vs_web.csv', index=False)

riqueza = d.dropna(subset=['alcohol', 'riqueza']).groupby('riqueza').agg(
    n=('alcohol', 'size'), pct_bebio=('alcohol', 'mean'), pct_diagnostico=(DESENLACE, 'mean'))
riqueza[['pct_bebio', 'pct_diagnostico']] *= 100
display(completo_vs_web)
print('Consumo de alcohol y diagnóstico reportado por quintil de riqueza (%):')
riqueza

,modelo,n,OR_alcohol,IC95_inf,IC95_sup,p
0,Notebook 02: modelo completo (14 predictores),8840,0.8055,0.6546,0.9912,0.0410
1,Este notebook: predictores del modelo web (9),8897,0.8263,0.6744,1.0124,0.0656
2,"Predictores web, misma muestra del notebook 02",8840,0.8260,0.6729,1.0139,0.0675
3,Web + presión arterial,8840,0.8248,0.6716,1.0129,0.0661
4,Web + nivel educativo,8840,0.8252,0.6716,1.0138,0.0673
5,Web + seguro de salud,8840,0.8197,0.6676,1.0066,0.0578
6,Web + índice de riqueza,8840,0.8067,0.6560,0.9919,0.0416


Consumo de alcohol y diagnóstico reportado por quintil de riqueza (%):


,n,pct_bebio,pct_diagnostico
riqueza,,,
1,184,65.2174,7.6087
2,1240,69.8387,3.9516
3,1959,73.5069,5.3088
4,2632,77.6596,6.4590
5,2893,82.8552,7.7774


**Resultado.** La diferencia no se debe al tamaño de la muestra (con la misma muestra, el modelo web da p = 0.068) sino a las variables de ajuste, y en particular al **índice de riqueza**: los hogares de mayor riqueza beben más y también reportan más diagnóstico. Al ajustar por riqueza, el OR del alcohol se aleja un poco de 1 y el valor p cruza el umbral de 0.05. Que la significancia dependa de incluir o no una variable de ajuste indica que la asociación residual del alcohol es **débil y sensible a la especificación del modelo**; ninguno de los dos valores debe presentarse como un hallazgo firme.

## 5. ¿Aporta el alcohol al modelo web? Validación cruzada con y sin alcohol

Mismo procedimiento del notebook 03: regresión logística con imputación y escalado dentro del pipeline, malla `C` ∈ {0.01, 0.1, 1, 10} y las mismas 5 particiones estratificadas (semilla 42). Criterio prefijado (protocolo 8.1): aporta de forma relevante si su exclusión reduce la AUC-ROC media en 0.005 o más, o empeora la calibración.

In [13]:
CONTINUAS = {'edad', 'imc', 'cintura', 'frutas_dias', 'verduras_dias'}


def pipeline_rl(predictores, C=None):
    continuas = [p for p in predictores if p in CONTINUAS]
    categoricas = [p for p in predictores if p not in CONTINUAS]
    prep = ColumnTransformer([
        ('continuas', Pipeline([('imputar', SimpleImputer(strategy='median')), ('escalar', StandardScaler())]), continuas),
        ('categoricas', Pipeline([('imputar', SimpleImputer(strategy='most_frequent')), ('escalar', StandardScaler())]), categoricas),
    ])
    return Pipeline([('prep', prep), ('modelo', LogisticRegression(max_iter=5000, **({'C': C} if C else {})))])


def calibracion(y_real, p):
    lp = np.log(p / (1 - p))
    pendiente = sm.Logit(y_real, sm.add_constant(lp)).fit(disp=False).params.iloc[1]
    intercepto = sm.GLM(y_real, np.ones(len(lp)), offset=lp, family=sm.families.Binomial()).fit().params.iloc[0]
    return pendiente, intercepto


y = d[DESENLACE].astype(int)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
variantes = {'con alcohol (modelo web)': PREDICTORES_WEB, 'sin alcohol': [p for p in PREDICTORES_WEB if p != 'alcohol']}

filas, oof, aucs_fold = [], {}, {}
for nombre, predictores in variantes.items():
    gs = GridSearchCV(pipeline_rl(predictores), {'modelo__C': [0.01, 0.1, 1, 10]}, scoring='roc_auc', cv=folds).fit(d[predictores], y)
    p = cross_val_predict(gs.best_estimator_, d[predictores], y, cv=folds, method='predict_proba')[:, 1]
    oof[nombre] = p
    aucs_fold[nombre] = np.array([gs.cv_results_[f'split{k}_test_score'][gs.best_index_] for k in range(5)])
    pendiente, intercepto = calibracion(y, p)
    filas.append({'variante': nombre, 'C': gs.best_params_['modelo__C'],
                  'AUC_ROC_media': gs.best_score_, 'AUC_ROC_de': gs.cv_results_['std_test_score'][gs.best_index_],
                  'AUC_PR_oof': average_precision_score(y, p), 'Brier_oof': brier_score_loss(y, p),
                  'pendiente_calibracion': pendiente, 'intercepto_calibracion': intercepto})
comparacion = pd.DataFrame(filas)
comparacion.to_csv(SALIDA / 'cv_con_sin_alcohol.csv', index=False)

dif_fold = aucs_fold['con alcohol (modelo web)'] - aucs_fold['sin alcohol']
print('Diferencia de AUC por partición (con - sin):', np.round(dif_fold, 4))
comparacion

Diferencia de AUC por partición (con - sin): [-0.0035 -0.0013  0.0018 -0.003   0.0023]


,variante,C,AUC_ROC_media,AUC_ROC_de,AUC_PR_oof,Brier_oof,pendiente_calibracion,intercepto_calibracion
0,con alcohol (modelo web),1,0.7815,0.0314,0.2241,0.0538,0.9823,0.0004
1,sin alcohol,1,0.7823,0.0325,0.2227,0.0538,0.9857,0.0004


In [14]:
repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEMILLA)
difs = []
for entrenamiento, prueba in repetida.split(d, y):
    aucs = {}
    for nombre, predictores in variantes.items():
        modelo = pipeline_rl(predictores, C=1).fit(d.iloc[entrenamiento][predictores], y.iloc[entrenamiento])
        aucs[nombre] = roc_auc_score(y.iloc[prueba], modelo.predict_proba(d.iloc[prueba][predictores])[:, 1])
    difs.append(aucs['con alcohol (modelo web)'] - aucs['sin alcohol'])
difs = np.array(difs)
pd.DataFrame({'diferencia_AUC_con_menos_sin': difs}).to_csv(SALIDA / 'cv_repetida_diferencias.csv', index=False)
print(f'10 x 5 validaciones cruzadas (C = 1): diferencia media de AUC (con - sin) = {difs.mean():+.4f} '
      f'(DE {difs.std(ddof=1):.4f}; percentiles 2.5-97.5: {np.percentile(difs, 2.5):+.4f} a {np.percentile(difs, 97.5):+.4f}); '
      f'la versión con alcohol fue mejor en {100 * (difs > 0).mean():.0f}% de las particiones')

10 x 5 validaciones cruzadas (C = 1): diferencia media de AUC (con - sin) = -0.0001 (DE 0.0020; percentiles 2.5-97.5: -0.0035 a +0.0024); la versión con alcohol fue mejor en 48% de las particiones


## 6. Conclusión

**1. No hay error de codificación ni de integración.** La pregunta `QS208` es la misma en los cinco años, y su codificación reproduce las reglas del cuestionario (incluido el salto de quienes nunca bebieron). La API construye exactamente la misma fila que el entrenamiento en los 8,897 registros comparados (diferencia máxima de probabilidad: 0.00009, por el redondeo del IMC).

**2. No hay error de interpretación de la contribución, pero sí un límite.** El modelo explica la clase positiva (diagnóstico reportado) y la contribución se mide respecto de la persona promedio de 2021–2024 (77% bebió en los últimos 12 meses). Por eso es negativa para quien bebió (−0.046 log-odds; equivale a multiplicar el odds por ≈ 0.96) y positiva para quien no bebió (+0.153). Es una contribución individual dentro del modelo, **no un efecto protector**.

**3. Es una asociación observada en los datos, explicada en gran parte por la edad y el sexo.**
- Sin ajustar, quienes bebieron en los últimos 12 meses reportan diagnóstico con menor frecuencia (5.5% frente a 9.0%; OR crudo 0.60).
- Las personas mayores y las mujeres beben menos y, a la vez, reportan más diagnóstico. Al ajustar solo por edad y sexo, el OR pasa a 0.84 (IC 95%: 0.70–1.03; p = 0.09).
- Con los predictores del modelo web, el OR es 0.83 (IC 95%: 0.67–1.01; p = 0.07): **no es estadísticamente significativo**. Con los 14 predictores del modelo completo es 0.81 (p = 0.04); la diferencia la produce el ajuste por índice de riqueza (sección 4.6), lo que muestra que la asociación residual es débil y sensible a la especificación.
- No hay evidencia de que la asociación cambie entre años, grupos de edad o sexo (pruebas de interacción, p ≥ 0.48).

**4. La causa de la asociación residual no puede determinarse con estos datos.** La hipótesis de cambio de hábitos tras el diagnóstico (causalidad inversa) es plausible, pero la ENDES no registra cuándo ni por qué alguien dejó de beber, ni la fecha del diagnóstico. Además, quienes bebieron alguna vez pero no en los últimos 12 meses no se diferencian de quienes nunca bebieron (OR ajustado 0.98; IC 95%: 0.55–1.77), aunque este grupo de referencia es pequeño (198 personas). Otras explicaciones posibles son el azar, la confusión residual o la clasificación errónea del desenlace por diabetes no diagnosticada. **Ninguna puede presentarse como comprobada.**

**5. El alcohol no aporta al modelo web.** Con las mismas particiones, el modelo sin alcohol tiene la misma discriminación (AUC-ROC media 0.782 frente a 0.782 con alcohol) y la misma calibración (Brier 0.0538 en ambos; pendientes 0.986 y 0.982). En 50 particiones (10 × 5), la diferencia media de AUC fue −0.0001 (percentiles 2.5–97.5: −0.004 a +0.002), y la versión con alcohol fue mejor en solo el 48% de las particiones. Según el criterio prefijado (reducción ≥ 0.005 o peor calibración), **el alcohol no aporta de forma relevante**.

**Implicaciones.** El modelo congelado se mantiene sin cambios y no se reentrena con 2025: su desempeño ya fue evaluado, y quitar el alcohol no modificaría la discriminación ni la calibración. El formulario conserva la pregunta, porque el modelo la usa, pero el sistema web no muestra la contribución del alcohol como factor, por ser pequeña, no significativa una vez ajustada y de causa desconocida. En futuras versiones del modelo, el alcohol podría excluirse de los predictores.